# Scrapping the ASI Conference Website: Chair of sessions and Plenary sessions
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [6]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Installing and loading all dependencies and libraries needed

In [2]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.7/481.7 kB 29.2 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,192 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,624 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnake

# Scrapping ISI: 3rd attempt: PDF MINER.SIX

## Extracting text

In [3]:
!pip install pdfminer.six
from pdfminer.high_level import extract_text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 55.3 MB/s eta 0:00:00


In [43]:
# Specify the path to your PDF in Google Drive
pdf_path = '/content/drive/My Drive/ISI/2018_removed.pdf'

In [44]:
text = extract_text(pdf_path)
print(text)


14th Annual Conference on Economic Growth and Development 
December 19-21, 2018 
Indian Statistical Institute, New Delhi 

Conference Programme 

Pre-conference Event: 18 December, Tuesday  

7:00 pm - 8:30 pm: Panel Discussion on "The Way forward for the Indian Economy" 

            (Auditorium, India Islamic Cultural Centre, 87-88, Lodhi Road, New Delhi - 110003) 

Panelists: 

K.P. Krishnan (Ministry of Skill Development and Entrepreneurship) 
T.N. Ninan (Business Standard) 
Ila Patnaik (National Institute of Public Finance and Policy) 
Sanjeev Sanyal (Ministry of Finance) 

Moderator: Ashok Kotwal (Ideas for India and University of British Columbia) 

            [The panel discussion is co-organized with the International Growth Centre and Ideas for India.] 

 
 
 
 
 
 
 
 
 
 
 
                                                                                                                                              
 
 
 
Day 1: 19 December, Wednesday  

9.00 - 10.10: Regis

Collect Plenary session chairs

In [45]:
def extract_plenary_sessions(text):
    """
    Extracts plenary session information into a structured DataFrame.
    """
    # Define regex patterns
    plenary_pattern = r"Plenary Session (\d+) \([^)]+\)\s+([\w\s.,;()]+)\(([^)]+)\):\s*“([^”]+)”"

    # Find all matches
    matches = re.findall(plenary_pattern, text)

    # Prepare structured data
    data = []
    for match in matches:
        session_number = match[0]
        presenter = match[1].strip()
        affiliation = match[2].strip()
        topic = match[3].strip()

        data.append({
            "Conference Year": 2019,
            "Plenary Session Number": session_number,
            "Presenter": presenter,
            "Affiliation": affiliation,
            "Topic": topic
        })

    # Convert to DataFrame
    return pd.DataFrame(data)

# Extract data
df = extract_plenary_sessions(text)

# Save or display the DataFrame
print(df)
df.to_csv("plenary_sessions.csv", index=False)


Empty DataFrame
Columns: []
Index: []


In [38]:
import re
import pandas as pd

def extract_sessions_and_panels(text):
    """
    Extracts plenary sessions and panel discussions into a structured DataFrame.
    """
    # Regex for plenary sessions
    plenary_pattern = r"Plenary Session (\d+) \([^)]+\)\s+([\w\s.,;()]+)\(([^)]+)\):\s*“([^”]+)”"

    # Regex for panel discussions
    panel_pattern = r"Panel Discussion on “([^”]+)” .*?Panelists:\s+([\s\S]+?)\n\s*\["

    data = []

    # Extract plenary sessions
    plenary_matches = re.findall(plenary_pattern, text)
    for match in plenary_matches:
        session_number = match[0]
        presenter = match[1].strip()
        affiliation = match[2].strip()
        topic = match[3].strip()

        data.append({
            "Conference Year": 2019,
            "Session Type": "Plenary Session",
            "Session Number": session_number,
            "Presenter": presenter,
            "Affiliation": affiliation,
            "Topic": topic
        })

    # Extract panel discussions
    panel_matches = re.findall(panel_pattern, text)
    for match in panel_matches:
        topic = match[0].strip()
        panelists = match[1].strip().split("\n")

        # Extract panelists and affiliations
        for panelist in panelists:
            panelist_match = re.match(r"([\w\s]+)\(([^)]+)\)", panelist.strip())
            if panelist_match:
                presenter = panelist_match[1].strip()
                affiliation = panelist_match[2].strip()

                data.append({
                    "Conference Year": 2023,
                    "Session Type": "Panel Discussion",
                    "Session Number": "N/A",  # Panels don't have session numbers
                    "Presenter": presenter,
                    "Affiliation": affiliation,
                    "Topic": topic
                })

    # Convert to DataFrame
    return pd.DataFrame(data)

# Extract data
df = extract_sessions_and_panels(text)

# Save or display the DataFrame
print(df)
df.to_csv("sessions_and_panels.csv", index=False)


Empty DataFrame
Columns: []
Index: []


Save the plenary sessions in the third sheet

In [34]:
import pandas as pd

# Install openpyxl if not already installed
!pip install openpyxl

# Define the file path in Google Colab
existing_file = '/content/drive/MyDrive/ISI/ISI2019.xlsx'

# Add the new sheet to the existing Excel file
with pd.ExcelWriter(existing_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    # Add the new DataFrame to the specified sheet name
    df.to_excel(writer, sheet_name='SPECIAL sessions Keynote, Plenary sessions', index=False)

# Confirm save location and display download link
print(f"Sheet 'SPECIAL sessions Keynote, Plenary sessions' has been successfully updated.")
print(f"The updated file is saved at: {existing_file}")


Sheet 'SPECIAL sessions Keynote, Plenary sessions' has been successfully updated.
The updated file is saved at: /content/drive/MyDrive/ISI/ISI2022.xlsx


/usr/local/lib/python3.10/dist-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
